In [1]:
import numpy as np
import os
import glob
import tensorflow as tf
import math
import pickle
#import matplotlib.pyplot as plt
from PIL import Image
from PIL import ImageFont
from PIL import ImageDraw

In [4]:
# Parameters
learning_rate = 0.001
training_iters = 500000
batch_size = 16
display_step = 100
SEED = 448

# Network Parameters
n_input = 9000 # MNIST data input (img shape: 50*60*3)
n_classes = len(signL.labels) # MNIST total classes (0-9 digits)
dropout = 0.75 # Dropout, probability to keep units

In [2]:
loadname = '../../class/t426.p'
class signLClass:
        def __init__(self):
                self.data = [np.zeros(1),np.zeros(1)]
                self.testdata = [np.zeros(1),np.zeros(1)]
                self.labels = []
                self.testlabels = []
                self.first = True
                self.testfirst = True
        def add_image(self, image, label):
                if self.first == False:
                        tmp_image = self.data[1]
                        tmp_label = self.data[0]
                self.data[1] = image
                self.data[0] = label
                #self.data[1] = np.reshape(self.data[1], [-1, 50, 60, 3])
                #self.data[0] = self.data[0].reshape((1,kind))
                if self.first == False:
                        self.data[1] = np.vstack((tmp_image, self.data[1]))
                        self.data[0] = np.vstack((tmp_label, self.data[0]))
                self.first = False
        def add_test_image(self, image, label):
                if self.testfirst == False:
                        tmp_image = self.testdata[1]
                        tmp_label = self.testdata[0]
                self.testdata[1] = image
                self.testdata[0] = label
                #self.data[1] = np.reshape(self.data[1], [-1, 50, 60, 3])
                #self.data[0] = self.data[0].reshape((1,kind))
                if self.testfirst == False:
                        self.testdata[1] = np.vstack((tmp_image, self.testdata[1]))
                        self.testdata[0] = np.vstack((tmp_label, self.testdata[0]))
                self.testfirst = False
        def train(self, n, batch_size):
                return signL.data[1][n:n+batch_size]
        def label(self, n, batch_size):
                return signL.data[0][n:n+batch_size]

def load_class(signL, loadname):
    signL = pickle.load(open(loadname, "rb"))
    return signL
signL = signLClass()
signL = load_class(signL, loadname)
print "Training data class named 'signL'. "

Training data class named 'signL'. 


In [13]:
# tf Graph input
x = tf.placeholder(tf.float32, [None, 50, 60, 3], name = 'placeholder_x')
y = tf.placeholder(tf.float32, [None, n_classes])
keep_prob = tf.placeholder(tf.float32, name = 'keep_prob') #dropout (keep probability)

In [14]:
x

<tf.Tensor 'placeholder_x_1:0' shape=(?, 50, 60, 3) dtype=float32>

In [15]:
# Create model
def conv2d(img, w, b):
    return tf.nn.relu(tf.nn.bias_add(tf.nn.conv2d(img, w,strides=[1, 1, 1, 1],padding='VALID'),b))

def max_pool(img, k):
    return tf.nn.max_pool(img, ksize=[1, k, k, 1],strides=[1, k, k, 1],padding='VALID')

In [16]:
# Store layers weight & bias

wc1 = tf.Variable(tf.random_normal([5, 5, 3, 64], stddev=1e-3)) # 5x5 conv, 1 input, 64 outputs
wc2 = tf.Variable(tf.random_normal([5, 5, 64, 64], stddev=1e-3)) # 5x5 conv, 64 inputs, 64 outputs
wd1 = tf.Variable(tf.random_normal([9*12*64, 1024], stddev=1e-3)) # fully connected, 7*7*64 inputs, 1024 outputs
wout = tf.Variable(tf.random_normal([1024, n_classes], stddev=1e-3)) # 1024 inputs, 10 outputs (class prediction)


bc1 = tf.Variable(tf.random_normal([64]))
bc2 = tf.Variable(tf.random_normal([64]))
bd1 = tf.Variable(tf.random_normal([1024]))
bout = tf.Variable(tf.random_normal([n_classes]))

In [17]:
# Construct model
_X = tf.reshape(x, shape=[-1, 50, 60, 3])


# Convolution Layer
conv1 = conv2d(_X,wc1,bc1)
# Max Pooling (down-sampling)
conv1 = max_pool(conv1, k=2)
# Apply Dropout
conv1 = tf.nn.dropout(conv1,keep_prob)

# Convolution Layer
conv2 = conv2d(conv1,wc2,bc2)
# Max Pooling (down-sampling)
conv2 = max_pool(conv2, k=2)
# Apply Dropout
conv2 = tf.nn.dropout(conv2, keep_prob)



In [18]:
# Fully connected layer
dense1 = tf.reshape(conv2, [-1, wd1.get_shape().as_list()[0]]) # Reshape conv2 output to fit dense layer input
dense1 = tf.nn.relu(tf.add(tf.matmul(dense1, wd1),bd1)) # Relu activation
dense1 = tf.nn.dropout(dense1, keep_prob) # Apply Dropout

# Output, class prediction
with tf.name_scope("pred"):
    pred = tf.add(tf.matmul(dense1, wout), bout)

#pred = conv_net(x, weights, biases, keep_prob)


regularizers = (tf.nn.l2_loss(wc1) + tf.nn.l2_loss(bc1)+(tf.nn.l2_loss(wc2) + tf.nn.l2_loss(bc2))+
                 (tf.nn.l2_loss(wd1) + tf.nn.l2_loss(bd1))+(tf.nn.l2_loss(wout) + tf.nn.l2_loss(bout)))

# Define loss and optimizer
cost = tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits(labels=y, logits=pred)) + 1e-6*regularizers
optimizer = tf.train.AdamOptimizer(learning_rate=learning_rate).minimize(cost)

# Evaluate model
correct_pred = tf.equal(tf.argmax(pred,1), tf.argmax(y,1))
accuracy = tf.reduce_mean(tf.cast(correct_pred, tf.float32))

In [ ]:
def get_pic(filename):
    img = Image.open(filename)
    img = img.resize((50,60), Image.BILINEAR)
    arr = np.array(img)
    ## make a 1-dimensional view of arr
    flat_arr = arr.ravel()
    flat_arr= flat_arr.reshape((1,9000))
    return flat_arr

In [ ]:
def get_label(filename):
    if filename[17]=='0':
        return np.array([1,0,0,0,0,0,0,0,0,0])
    if filename[17]=='1':
        return np.array([0,1,0,0,0,0,0,0,0,0])
    if filename[17]=='2':
        return np.array([0,0,1,0,0,0,0,0,0,0])
    if filename[17]=='3':
        return np.array([0,0,0,1,0,0,0,0,0,0])
    if filename[17]=='4':
        return np.array([0,0,0,0,1,0,0,0,0,0])
    if filename[17]=='5':
        return np.array([0,0,0,0,0,1,0,0,0,0])
    if filename[17]=='6':
        return np.array([0,0,0,0,0,0,1,0,0,0])
    if filename[17]=='7':
        return np.array([0,0,0,0,0,0,0,1,0,0])
    if filename[17]=='8':
        return np.array([0,0,0,0,0,0,0,0,1,0])
    if filename[17]=='9':
        return np.array([0,0,0,0,0,0,0,0,0,1])

In [ ]:
def tget_label(filename):
    if filename[16]=='0':
        return np.array([1,0,0,0,0,0,0,0,0,0])
    if filename[16]=='1':
        return np.array([0,1,0,0,0,0,0,0,0,0])
    if filename[16]=='2':
        return np.array([0,0,1,0,0,0,0,0,0,0])
    if filename[16]=='3':
        return np.array([0,0,0,1,0,0,0,0,0,0])
    if filename[16]=='4':
        return np.array([0,0,0,0,1,0,0,0,0,0])
    if filename[16]=='5':
        return np.array([0,0,0,0,0,1,0,0,0,0])
    if filename[16]=='6':
        return np.array([0,0,0,0,0,0,1,0,0,0])
    if filename[16]=='7':
        return np.array([0,0,0,0,0,0,0,1,0,0])
    if filename[16]=='8':
        return np.array([0,0,0,0,0,0,0,0,1,0])
    if filename[16]=='9':
        return np.array([0,0,0,0,0,0,0,0,0,1])

In [ ]:
def get_rank_pic(path,i):
    #print i
    first = True
    for filename in glob.glob(os.path.join(path, '*.png'))[i:i+batch_size]:
        if first==False:
            rank_arr = flat_arr
        flat_arr = get_pic(filename)
        if first==False:
            flat_arr = np.vstack((rank_arr, flat_arr))
        first = False
    return flat_arr

In [ ]:
def get_rank_lable(path,i):
    first = True
    for filename in glob.glob(os.path.join(path, '*.png'))[i:i+batch_size]:
        if first==False:
            rank_label = flat_label
        flat_label = get_label(filename)
        flat_label = flat_label.reshape((1,10))
        if first==False:
            flat_label = np.vstack((rank_label, flat_label))
        first = False
    return flat_label

In [ ]:
def tget_rank_lable(path,i):
    first = True
    for filename in glob.glob(os.path.join(path, '*.png'))[i:i+batch_size]:
        if first==False:
            rank_label = flat_label
        flat_label = tget_label(filename)
        flat_label = flat_label.reshape((1,10))
        if first==False:
            flat_label = np.vstack((rank_label, flat_label))
        first = False
    return flat_label

In [19]:
# Initializing the variables
init = tf.global_variables_initializer()

In [20]:
signL.train(0, 10).shape

(10, 50, 60, 3)

In [21]:
# Launch the graph
path = 'rgbdata/training'
saver = tf.train.Saver()
sess=tf.Session()
sess.run(init)
step = 1
# Keep training until reach max iterations
while step * batch_size < training_iters:
    batch_xs = signL.train(step, batch_size)
    batch_ys = signL.label(step, batch_size)
    # Fit training using batch data
    sess.run(optimizer, feed_dict={x: batch_xs, y: batch_ys, keep_prob: 0.4, learning_rate: 1e-5})
    if step % display_step == 0 :
        # Calculate batch accuracy
        acc = sess.run(accuracy, feed_dict={x: batch_xs, y: batch_ys, keep_prob: 0.4, learning_rate: 1e-5})
        # Calculate batch loss
        loss = sess.run(cost, feed_dict={x: batch_xs, y: batch_ys, keep_prob: 1.})
        print "Iter " + str(step*batch_size) + ", Minibatch Loss= " + "{:.6f}".format(loss) + ", Training Accuracy= " + "{:.5f}".format(acc)
    step += 1
print "Optimization Finished!"
# Test model
test_images = signL.testdata[1]
test_labels = signL.testdata[0]
# Calculate accuracy for 256 mnist test images
print "Testing Accuracy:", sess.run(accuracy, feed_dict={x: test_images, y: test_labels, keep_prob: 1.})

TypeError: Cannot interpret feed_dict key as Tensor: Can not convert a float into a Tensor.

In [ ]:
saver.save(sess, 'Model-Demo')

In [ ]:
def testing(filename):
    stest_images = get_pic(filename)
    print sess.run(tf.argmax(pred,1), feed_dict={x: stest_images, keep_prob: 1.})

In [ ]:
def testing_all(path):
    for filename in glob.glob(os.path.join(path, '*.png')):
        print filename
        stest_images = get_pic(filename)
        print sess.run(tf.argmax(pred,1), feed_dict={x: stest_images, keep_prob: 1.})

In [ ]:
import tensorflow as tf; print(tf.__version__)

In [ ]:
filename = 'rgbdata/testing/6_491.png'
testing(filename)

In [ ]:
path = 'rgbdata/testing'
testing_all(path)

In [ ]:
sess = tf.Session()
new_saver = tf.train.import_meta_graph('Model-Demo.meta')
new_saver.restore(sess, tf.train.latest_checkpoint('./'))
all_vars = tf.trainable_variables()
